In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import torch.optim as optim
import copy
import pandas as pd 
import torch.nn.functional as F
import numpy as np
import cv2 # For heatmap visualization
from PIL import Image
from torchvision import transforms
import os

In [ ]:
def get_device():
    """
    Returns the best available PyTorch device:
    1. CUDA (GPU)
    2. MPS (Apple Silicon GPU)
    3. CPU
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    else:
        return torch.device("cpu")


# Example usage:
device = get_device()
print("Using device:", device)

In [ ]:
TRAIN_ROOT = "data/brain_mri/training"
TEST_ROOT = "data/brain_mri/testing"
train_dataset = torchvision.datasets.ImageFolder(root=TRAIN_ROOT)
test_dataset = torchvision.datasets.ImageFolder(root=TRAIN_ROOT)

In [ ]:
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        self.vgg16 = models.vgg16(pretrained=True) 

        # Replace output layer according to our problem
        in_feats = self.vgg16.classifier[6].in_features 
        # We replace the original classifier[6] with a sequence that includes Dropout.
        self.vgg16.classifier = nn.Sequential(
            *self.vgg16.classifier[:-1], # Keep layers 0 through 5 (which includes 3 Linear layers)
            nn.Dropout(p=0.4),           # ADD DROPOUT HERE (p=0.4 is a good starting point)
            nn.Linear(in_feats, 4)       # Your final 4-class classification layer
        )

    def forward(self, x):
        x = self.vgg16(x)
        return x

model = CNNModel()
model.to(device)
model

In [ ]:
train_dataset = torchvision.datasets.ImageFolder(
        root=TRAIN_ROOT,
        transform=transforms.Compose([
                      transforms.Resize((255,255)),
                      transforms.ToTensor()
        ])
)

test_dataset = torchvision.datasets.ImageFolder(
        root=TEST_ROOT,
        transform=transforms.Compose([
                      transforms.Resize((255,255)),
                      transforms.ToTensor()
        ])
)

In [ ]:
batch_size = 32
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=True
)

In [ ]:
cross_entropy_loss = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001, weight_decay=1e-4)
epochs = 10

In [ ]:
for epoch in range(epochs):  
    for i, batch in enumerate(train_loader, 0):
        inputs, labels = batch
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        # Labels are automatically one-hot-encoded
        loss = cross_entropy_loss(outputs, labels)
        loss.backward()
        optimizer.step()
        print(loss)


In [ ]:
torch.save(model, 'brain_tumor_full_model.pth')

In [ ]:
inputs, labels = next(iter(test_loader))
inputs = inputs.to(device)
labels = labels.numpy()
outputs = model(inputs).max(1).indices.detach().cpu().numpy()
comparison = pd.DataFrame()
print("Batch accuracy: ", (labels==outputs).sum()/len(labels))
comparison["labels"] = labels

comparison["outputs"] = outputs
comparison

In [ ]:
def new_layer(layer, g):
    """Clone a layer and pass its parameters through the function g."""
    layer = copy.deepcopy(layer)
    try: layer.weight = torch.nn.Parameter(g(layer.weight))
    except AttributeError: pass
    try: layer.bias = torch.nn.Parameter(g(layer.bias))
    except AttributeError: pass
    return layer


In [ ]:
def dense_to_conv(layers):
    """ Converts a dense layer to a conv layer """
    newlayers = []
    for i,layer in enumerate(layers):
        if isinstance(layer, nn.Linear):
            newlayer = None
            if i == 0:
                m, n = 512, layer.weight.shape[0]
                newlayer = nn.Conv2d(m,n,7)
                newlayer.weight = nn.Parameter(layer.weight.reshape(n,m,7,7))
            else:
                m,n = layer.weight.shape[1],layer.weight.shape[0]
                newlayer = nn.Conv2d(m,n,1)
                newlayer.weight = nn.Parameter(layer.weight.reshape(n,m,1,1))
            newlayer.bias = nn.Parameter(layer.bias)
            newlayers += [newlayer]
        else:
            newlayers += [layer]
    return newlayers


In [ ]:
def get_linear_layer_indices(model):
    offset = len(model.vgg16._modules['features']) + 1
    indices = []
    for i, layer in enumerate(model.vgg16._modules['classifier']): 
        if isinstance(layer, nn.Linear): 
            indices.append(i)
    indices = [offset + val for val in indices]
    return indices


In [ ]:
def apply_lrp_on_vgg16(model, image):
    image = torch.unsqueeze(image, 0)
    # >>> Step 1: Extract layers
    layers = list(model.vgg16._modules['features']) \
                + [model.vgg16._modules['avgpool']] \
                + dense_to_conv(list(model.vgg16._modules['classifier']))
    linear_layer_indices = get_linear_layer_indices(model)
    # >>> Step 2: Propagate image through layers and store activations
    n_layers = len(layers)
    activations = [image] + [None] * n_layers # list of activations
    
    for layer in range(n_layers):
        if layer in linear_layer_indices:
            if layer == 32:
                activations[layer] = activations[layer].reshape((1, 512, 7, 7))
        activation = layers[layer].forward(activations[layer])
        if isinstance(layers[layer], torch.nn.modules.pooling.AdaptiveAvgPool2d):
            activation = torch.flatten(activation, start_dim=1)
        activations[layer+1] = activation

    # >>> Step 3: Replace last layer with one-hot-encoding
    output_activation = activations[-1].detach().cpu().numpy()
    max_activation = output_activation.max()
    one_hot_output = [val if val == max_activation else 0 
                        for val in output_activation[0]]

    activations[-1] = torch.FloatTensor([one_hot_output]).to(device)

    # >>> Step 4: Backpropagate relevance scores
    relevances = [None] * n_layers + [activations[-1]]
    # Iterate over the layers in reverse order
    for layer in range(0, n_layers)[::-1]:
        current = layers[layer]
        # Treat max pooling layers as avg pooling
        if isinstance(current, torch.nn.MaxPool2d):
            layers[layer] = torch.nn.AvgPool2d(2)
            current = layers[layer]
        if isinstance(current, torch.nn.Conv2d) or \
           isinstance(current, torch.nn.AvgPool2d) or\
           isinstance(current, torch.nn.Linear):
            activations[layer] = activations[layer].data.requires_grad_(True)
            
            # Apply variants of LRP depending on the depth
            # see: https://link.springer.com/chapter/10.1007%2F978-3-030-28954-6_10
            # Lower layers, LRP-gamma >> Favor positive contributions (activations)
            if layer <= 16:       rho = lambda p: p + 0.25*p.clamp(min=0); incr = lambda z: z+1e-9
            # Middle layers, LRP-epsilon >> Remove some noise / Only most salient factors survive
            if 17 <= layer <= 30: rho = lambda p: p;                       incr = lambda z: z+1e-9+0.25*((z**2).mean()**.5).data
            # Upper Layers, LRP-0 >> Basic rule
            if layer >= 31:       rho = lambda p: p;                       incr = lambda z: z+1e-9
            
            # Transform weights of layer and execute forward pass
            z = incr(new_layer(layers[layer],rho).forward(activations[layer]))
            # Element-wise division between relevance of the next layer and z
            s = (relevances[layer+1]/z).data                                     
            # Calculate the gradient and multiply it by the activation
            (z * s).sum().backward(); 
            c = activations[layer].grad       
            # Assign new relevance values           
            relevances[layer] = (activations[layer]*c).data                          
        else:
            relevances[layer] = relevances[layer+1]

    # >>> Potential Step 5: Apply different propagation rule for pixels
    return relevances[0]

In [ ]:
# Calculate relevances for first image in this test batch
image_id = 31
image_relevances = apply_lrp_on_vgg16(model, inputs[image_id])
image_relevances = image_relevances.permute(0,2,3,1).detach().cpu().numpy()[0]
image_relevances = np.interp(image_relevances, (image_relevances.min(),
                                                image_relevances.max()), 
                                                (0, 1))

In [ ]:
pred_label = list(test_dataset.class_to_idx.keys())[
             list(test_dataset.class_to_idx.values())
            .index(labels[image_id])]
if outputs[image_id] == labels[image_id]:
    print("Groundtruth for this image: ", pred_label)

    # Plot images next to each other
    plt.axis('off')
    plt.subplot(1,2,1)
    plt.imshow(image_relevances[:,:,0], cmap="seismic")
    plt.subplot(1,2,2)
    plt.imshow(inputs[image_id].permute(1,2,0).detach().cpu().numpy())
    plt.show()
else:
    print("This image is not classified correctly.")

## Using Grad-CAM to explain the model

In [ ]:
import torch
import numpy as np
import cv2 # Ensure you have opencv-python installed: pip install opencv-python

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        # NOTE: self.model.eval() is kept here for initialization,
        # but is reinforced in __call__ for stability.
        self.model.eval()
        
        self.target_layer = target_layer
        
        # Hooks to store activations and gradients
        self.gradients = None
        self.activations = None
        
        # Register hooks
        self.target_layer.register_forward_hook(self._save_activations)
        # Using register_full_backward_hook to be compatible with latest PyTorch
        self.target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, input, output):
        # Save feature map
        self.activations = output

    def _save_gradients(self, module, grad_input, grad_output):
        # Save gradients w.r.t. the output of the target layer
        self.gradients = grad_output[0] 

    # target_category is NO LONGER optional and MUST be provided
    def __call__(self, input_tensor, target_category):
        # 1. Enforce Evaluation Mode for stability (Crucial!)
        self.model.eval() 
        
        # Check for target_category argument
        if target_category is None:
             raise ValueError("target_category must be provided to force the stable prediction index.")

        # 2. Forward Pass
        # The forward hook saves the activations automatically
        output = self.model(input_tensor) 
        
        # 3. Zero gradients and perform backward pass for the target class
        self.model.zero_grad()
        
        # Create the one-hot vector for the *forced* target class
        one_hot = torch.zeros_like(output)
        one_hot[0][target_category] = 1 
        
        # Backpropagate only the gradient of the target class
        output.backward(gradient=one_hot, retain_graph=True)

        # 4. Get Activations and Gradients
        activations = self.activations.data.squeeze(0) # (C, H, W)
        gradients = self.gradients.data.squeeze(0)      # (C, H, W)
        
        # 5. Compute the 'alpha' weights: Global Average Pooling of Gradients
        weights = torch.mean(gradients, dim=(1, 2), keepdim=True) # (C, 1, 1)

        # 6. Compute the Grad-CAM (weighted sum of activations)
        cam = (weights * activations).sum(dim=0).relu()
        
        # 7. Normalize and Upscale the Heatmap
        cam = cam.cpu().numpy()
        # Normalize 0-1
        cam = (cam - np.min(cam)) / (np.max(cam) - np.min(cam)) if np.max(cam) > np.min(cam) else cam * 0.0
        
        # Resize to input image size (assuming square input)
        (h, w) = input_tensor.shape[-2:]
        cam = cv2.resize(cam, (w, h))

        # Return the heatmap and the target_category used (which is the stable index)
        return cam, target_category

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import cv2

def visualize_gradcam(original_input_tensor: torch.Tensor, 
                      heatmap_numpy: np.ndarray, 
                      predicted_class_idx: int,
                      class_names: list,
                      target_class: str,
                      output_filename: str = None):
    """
    Visualizes the Grad-CAM heatmap overlayed on the original image.

    Args:
        original_input_tensor (torch.Tensor): The pre-processed, normalized input tensor.
        heatmap_numpy (np.ndarray): The 0-1 normalized Grad-CAM heatmap (resized to input size).
        predicted_class_idx (int): The index of the class the model predicted.
        class_names (list): List of class names (directory names).
        target_class (str): The TRUE label of the image (for context in the title).
        output_filename (str): Optional filename to save the plot.
    """
    
    # 1. Reverse Normalization (to view the image correctly)
    
    # Standard ImageNet normalization values (assumed)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    # Move tensor to CPU and convert to numpy
    img_np = original_input_tensor.cpu().squeeze(0).numpy().transpose((1, 2, 0)) # C, H, W -> H, W, C

    # Reverse normalization:
    img_np = img_np * std + mean
    img_np = np.clip(img_np, 0, 1) # Clip values to [0, 1] for display

    # 2. Convert Heatmap to RGB
    # Convert heatmap to 3-channel image and apply color mapping (COLORMAP_JET is good for heatmaps)
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_numpy), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB) # Convert BGR to RGB

    # 3. Create the Overlay (Blended Image)
    # Overlay the heatmap on the original image (Alpha blending: 0.4*heatmap + 0.6*image)
    overlay = cv2.addWeighted(np.uint8(255 * img_np), 0.6, heatmap_colored, 0.4, 0)
    overlay = overlay / 255.0 # Normalize back to 0-1 for matplotlib display

    # 4. Plotting the Results
    
    predicted_class_name = class_names[predicted_class_idx]
    
    plt.figure(figsize=(15, 5))
    plt.suptitle(f"Grad-CAM Analysis | TRUE: {target_class} | PREDICTED (Stable): {predicted_class_name}", 
                 fontsize=14, fontweight='bold')

    # Original Image
    plt.subplot(1, 3, 1)
    plt.imshow(img_np)
    plt.title("Original Image (Normalized)")
    plt.axis('off')

    # Heatmap
    plt.subplot(1, 3, 2)
    plt.imshow(heatmap_numpy, cmap='jet')
    plt.title("Grad-CAM Heatmap")
    plt.axis('off')

    # Overlay
    plt.subplot(1, 3, 3)
    plt.imshow(overlay)
    plt.title("Heatmap Overlay")
    plt.axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjust layout for suptitle

    if output_filename:
        plt.savefig(output_filename, dpi=300)
        print(f"Plot saved to {output_filename}")
        
    plt.show()

In [ ]:
import torch
from PIL import Image

def get_stable_prediction_index(model, image_path, transforms, device):
    """
    Performs a stable inference (model.eval() and torch.no_grad()) on a single image 
    to get the predicted class index.
    """
    
    # --- Prepare Input Tensor ---
    
    # 1. Load the image
    img = Image.open(image_path).convert('RGB')
    
    # 2. Apply the necessary transforms (Resize, ToTensor, Normalize)
    input_tensor = transforms(img)
    
    # 3. Add a batch dimension (VGG-16 expects [Batch, Channels, Height, Width])
    input_tensor = input_tensor.unsqueeze(0) 
    
    # 4. Move the tensor to the correct device
    input_tensor = input_tensor.to(device)
    
    # --- Stable Inference ---
    
    # Crucial 1: Set model to evaluation mode (turns off Dropout)
    model.eval()
    
    # Crucial 2: Disable gradient calculation
    with torch.no_grad():
        # Forward pass
        output_logits = model(input_tensor)
        
        # Get the index of the highest logit
        stable_pred_idx = torch.argmax(output_logits, dim=1).item()
        
    # Return the prepared tensor and the stable index
    return input_tensor, stable_pred_idx

# --- Example Execution ---

# Define your image path (e.g., a path from your CSV)
# TARGET_IMAGE_PATH = 'data/brain_mri/giloma_tumor/G_001.jpg' 

# Get the necessary inputs for Grad-CAM
# image_tensor, stable_pred_idx = get_stable_prediction_index(
#     model=model, 
#     image_path=TARGET_IMAGE_PATH, 
#     transforms=test_transforms, 
#     device=DEVICE
# )

# # This index is now guaranteed to be the stable prediction index
# print(f"Stable Prediction Index: {stable_pred_idx}")
# print(f"Stable Predicted Class: {CLASS_NAMES[stable_pred_idx]}")

# # You now pass these two variables to your Grad-CAM function:
# heatmap, target_used = grad_cam(
#     input_tensor=image_tensor, 
#     target_category=stable_pred_idx 
# )

In [ ]:
# Define the standard VGG-16 preprocessing steps
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    # The crucial normalization for VGG-16
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# 1. Load the image using PIL
giloma_img = Image.open("data/brain_mri/Testing/glioma_tumor/image(22).jpg")

# 2. Apply transformations
giloma_image_tensor_chw = preprocess(giloma_img) # Shape: (C, H, W)

# 3. Define the image_tensor (Add batch dimension and move to device)
giloma_image_tensor = giloma_image_tensor_chw.unsqueeze(0).to(device) # Shape: (1, C, H, W)

In [ ]:
# Define your image path (e.g., a path from your CSV)
TARGET_IMAGE_PATH = "data/brain_mri/Testing/glioma_tumor/image(22).jpg"

# Get the necessary inputs for Grad-CAM
image_tensor, stable_pred_idx = get_stable_prediction_index(
    model=model, 
    image_path=TARGET_IMAGE_PATH, 
    transforms=preprocess, 
    device=device
)

# # This index is now guaranteed to be the stable prediction index
# print(f"Stable Prediction Index: {stable_pred_idx}")
# print(f"Stable Predicted Class: {CLASS_NAMES[stable_pred_idx]}")

# # You now pass these two variables to your Grad-CAM function:
# heatmap, target_used = grad_cam(
#     input_tensor=image_tensor, 
#     target_category=stable_pred_idx 
# )

In [ ]:
grad_cam = GradCAM(model=model, target_layer=model.vgg16.features[30])

In [ ]:
heatmap, target_used = grad_cam(
    input_tensor=image_tensor.to(device), 
    target_category=stable_pred_idx # <--- FORCING THE STABLE INDEX
)

print(f"Model predicted class: {predicted_class_idx}")

In [ ]:
# Assuming these variables are defined from the stable prediction step:
# giloma_image_tensor (the prepared input tensor)
# heatmap (the result from grad_cam(..., target_category=stable_pred_idx))
# stable_pred_idx (the stable prediction index)
# CLASS_NAMES (the list of class names)

# The target class is the TRUE class label for the image being tested (e.g., 'giloma_tumor')
TRUE_TARGET_CLASS = "giloma" # Assuming your giloma image is index 0

visualize_gradcam(
    original_input_tensor=giloma_image_tensor, 
    heatmap_numpy=heatmap, 
    predicted_class_idx=stable_pred_idx, # Use the stable index!
    class_names=test_dataset.classes,
    target_class=TRUE_TARGET_CLASS,
    output_filename='gradcam_giloma_fixed.png'
)

In [ ]:
from torchmetrics.functional import confusion_matrix
# NOTE: While torchmetrics is an external library, it's widely used in PyTorch projects
# for metrics, and is often preferred over writing the entire confusion matrix logic manually.
# If you must strictly use ONLY torch and Python standard library, you'd implement 
# the confusion matrix manually.

def calculate_pytorch_metrics(model, data_loader, device, num_classes, class_names):
    """
    Calculates and prints the per-class performance metrics using PyTorch tensors.

    Args:
        model (nn.Module): The trained PyTorch model.
        data_loader (DataLoader): The DataLoader for the test/validation set.
        device (torch.device): The device (CPU or CUDA) the model is on.
        num_classes (int): The total number of classes (e.g., 4).
        class_names (list): A list of class names.
    """
    model.eval()
    all_true_labels = []
    all_predictions = []

    # 1. Collect all predictions and true labels
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            
            # Get the predicted class indices
            _, predicted_indices = torch.max(outputs, 1)

            all_true_labels.append(labels.cpu())
            all_predictions.append(predicted_indices.cpu())

    # Concatenate all lists into single tensors
    true_labels = torch.cat(all_true_labels)
    predictions = torch.cat(all_predictions)

    # 2. Calculate the Confusion Matrix
    # The confusion matrix C is where C[i][j] is the number of samples
    # known to be in class i and predicted to be in class j.
    conf_matrix = confusion_matrix(predictions, true_labels, task='multiclass', num_classes=num_classes)
    
    # 3. Calculate Performance Metrics from the Confusion Matrix
    
    print("\n--- Pure PyTorch Per-Class Metrics ---")
    print("Class | Precision | Recall | F1-Score | Support")
    print("-------------------------------------------------")

    for i in range(num_classes):
        # True Positives (TP): Correctly predicted class i
        TP = conf_matrix[i, i].item()
        
        # False Positives (FP): Predicted class i, but the true class was not i (sum of column i excluding TP)
        FP = conf_matrix[:, i].sum().item() - TP
        
        # False Negatives (FN): True class was i, but predicted as something else (sum of row i excluding TP)
        FN = conf_matrix[i, :].sum().item() - TP
        
        # Support (Actual count for class i)
        support = conf_matrix[i, :].sum().item()

        # Calculate metrics
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        print(f"{class_names[i]:<5} | {precision:.4f} | {recall:.4f} | {f1_score:.4f} | {support:<7}")
        
    print("-------------------------------------------------")
    
    
# NOTE: If you don't want to use torchmetrics, you can calculate the confusion matrix 
# manually using loops or PyTorch's `scatter_add_` function, but it is complex.

In [ ]:
calculate_pytorch_metrics(
    model=model,
    data_loader=test_loader,
    device=device,
    num_classes=len(test_dataset.classes),
    class_names=test_dataset.classes
)

In [ ]:
import matplotlib.pyplot as plt
from torchmetrics.classification import MulticlassAUROC

def plot_multiclass_auroc(model, data_loader, device, num_classes, class_names):
    """
    Calculates and prints the macro and weighted average AUC for a multi-class model.

    Args:
        model (nn.Module): The trained PyTorch model.
        data_loader (DataLoader): The DataLoader for the test/validation set.
        device (torch.device): The device (CPU or CUDA) the model is on.
        num_classes (int): The total number of classes (e.g., 4).
        class_names (list): A list of class names.
    """
    model.eval()
    all_true_labels = []
    all_logits = [] # Collect raw model outputs (logits)

    print("--- Starting AUC Calculation ---")
    
    # 1. Collect all labels and logits
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            # Forward pass: outputs are the raw logits
            outputs = model(inputs)

            all_true_labels.append(labels.cpu())
            all_logits.append(outputs.cpu())

    # Concatenate all lists into single tensors
    true_labels = torch.cat(all_true_labels)
    logits = torch.cat(all_logits)

    # 2. Calculate AUC Scores using MulticlassAUROC
    
    # Initialize AUROC metric for Macro Average
    auroc_macro = MulticlassAUROC(
        num_classes=num_classes, 
        average='macro'
    )
    
    # Initialize AUROC metric for Weighted Average
    auroc_weighted = MulticlassAUROC(
        num_classes=num_classes, 
        average='weighted'
    )
    
    # Initialize AUROC metric for Per-Class Scores (used for plotting)
    auroc_none = MulticlassAUROC(
        num_classes=num_classes, 
        average='none' # 'none' returns score for each cla
    )

    # Compute the scores
    macro_auroc = auroc_macro(logits, true_labels)
    weighted_auroc = auroc_weighted(logits, true_labels)
    per_class_auroc = auroc_none(logits, true_labels)
    
    # 3. Print Results
    print("\nAUC Scores:")
    print(f"  Macro Average AUC:   {macro_auroc.item():.4f} (Treats all classes equally)")
    print(f"  Weighted Average AUC: {weighted_auroc.item():.4f} (Weights classes by support/count)")
    
    print("\nPer-Class AUC Scores:")
    for name, score in zip(class_names, per_class_auroc):
        print(f"  - {name:<20}: {score:.4f}")
        
    # 4. Plotting the results
    
    # Create labels for the bar chart
    labels = class_names + ['Macro Avg', 'Weighted Avg']
    scores = per_class_auroc.tolist() + [macro_auroc.item(), weighted_auroc.item()]
    colors = ['skyblue'] * num_classes + ['gold', 'coral']

    plt.figure(figsize=(10, 6))
    plt.bar(labels, scores, color=colors)
    plt.axhline(y=0.5, color='r', linestyle='--', label='Random Guess (AUC=0.5)')
    
    plt.ylim(0.0, 1.0) # AUC range is 0 to 1
    plt.ylabel("AUC Score")
    plt.title("Multiclass Area Under the ROC Curve (AUROC)")
    plt.xticks(rotation=45, ha='right')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Assuming you have already defined these variables ---
# model = ... (your loaded and trained CNNModel)
# TEST_LOADER = ... (your DataLoader for the test set)
# DEVICE = ... (torch.device)
# NUM_CLASSES = 4
# CLASS_NAMES = ['giloma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumour']

plot_multiclass_auroc(
    model=model,
    data_loader=test_loader,
    device=device,
    num_classes=len(test_dataset.classes),
    class_names=test_dataset.classes
)

In [ ]:
# --- NEW: Get the file paths from the original ImageFolder dataset ---
# You MUST run this *after* creating your test_dataset:
# test_dataset = torchvision.datasets.ImageFolder(root=TEST_ROOT, transform=test_transforms)
def get_filepaths(image_folder_dataset):
    """
    Extracts the list of file paths from the ImageFolder dataset.
    The order of files in this list corresponds to the indices used by the DataLoader.
    """
    return [s[0] for s in image_folder_dataset.samples]


def export_predictions_to_csv_with_paths(model, data_loader, device, class_names, all_filepaths, output_filename='predictions_analysis_with_paths.csv'):
    """
    Runs the model and exports a CSV with Target Class, Predicted Class, and the File Path.

    Args:
        model (nn.Module): The trained PyTorch model.
        data_loader (DataLoader): The DataLoader for the test/validation set.
        device (torch.device): The device (CPU or CUDA) the model is on.
        class_names (list): A list of class names in the order of their indices.
        all_filepaths (list): The list of all file paths corresponding to the samples in the DataLoader.
        output_filename (str): The name of the CSV file to save.
    """
    model.eval()
    
    # Lists to store the results
    results = {
        'File_Path': [],
        'Target_Class': [],
        'Predicted_Class': []
    }
    
    current_index = 0 # Track the index of the sample in the entire dataset

    print("--- Starting prediction export with file paths ---")
    
    with torch.no_grad():
        for inputs, labels in data_loader:
            batch_size = inputs.size(0)
            
            inputs = inputs.to(device)
            
            # Forward pass
            outputs = model(inputs)
            _, predicted_indices = torch.max(outputs, 1)

            # Move true labels and predictions to CPU
            true_labels_cpu = labels.cpu().numpy()
            predicted_indices_cpu = predicted_indices.cpu().numpy()

            # Process the batch results
            for i in range(batch_size):
                sample_index = current_index + i
                true_idx = true_labels_cpu[i]
                pred_idx = predicted_indices_cpu[i]
                
                # Append the data
                results['File_Path'].append(all_filepaths[sample_index])
                results['Target_Class'].append(class_names[true_idx])
                results['Predicted_Class'].append(class_names[pred_idx])
                
            current_index += batch_size

    # 2. Create and Save the DataFrame
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_filename, index=False)

    # 3. Print confirmation
    print(f"\n✅ Successfully exported {len(df_results)} predictions with paths to: {os.path.abspath(output_filename)}")

In [ ]:
export_predictions_to_csv_with_paths(model=model,
                          data_loader=test_loader,
                          device=device,
                          all_filepaths=get_filepaths(test_dataset),
                          class_names=test_dataset.classes
                          )